# Parte 1 — Extração de dados climáticos (Open-Meteo)

**Objetivo desta parte:** aprender a consumir uma API pública com `requests`, entender a
forma do JSON retornado (arrays paralelos por variável) e salvar o dado bruto em disco
antes de qualquer tratamento.

Vamos usar a **Historical Weather API** do [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api),
que devolve dados horários já reconstruídos (reanálise ERA5) para qualquer ponto do
planeta — não exige API key, o que é ótimo para fins didáticos.

Escolhemos **5 cidades brasileiras** com climas bem diferentes entre si, para que as
próximas partes já nasçam com uma dimensão de comparação interessante:

| Cidade | Região | Clima esperado |
|---|---|---|
| São Paulo | Sudeste | Subtropical, ameno |
| Rio de Janeiro | Sudeste | Tropical, litorâneo |
| Manaus | Norte | Equatorial, quente e úmido |
| Porto Alegre | Sul | Subtropical, mais frio |
| Recife | Nordeste | Tropical, litorâneo, seco/chuvoso bem marcado |


In [1]:
import json
import time
from pathlib import Path
import requests


## 1. Definindo cidades e período
Introduzir o básico de api

Fixamos um período histórico (janeiro de 2025 — verão no Brasil) em vez de "últimos N
dias" para que o notebook seja **reprodutível**: rodando hoje ou daqui a um mês, o
resultado é o mesmo.


In [2]:
CIDADES = {
    "sao_paulo": {"nome_exibicao": "São Paulo", "lat": -23.5505, "lon": -46.6333},
    "rio_de_janeiro": {"nome_exibicao": "Rio de Janeiro", "lat": -22.9068, "lon": -43.1729},
    "manaus": {"nome_exibicao": "Manaus", "lat": -3.1190, "lon": -60.0217},
    "porto_alegre": {"nome_exibicao": "Porto Alegre", "lat": -30.0346, "lon": -51.2177},
    "recife": {"nome_exibicao": "Recife", "lat": -8.0476, "lon": -34.8770},
}

DATA_INICIO = "2025-01-01"
DATA_FIM = "2025-01-31"

VARIAVEIS_HORARIAS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
]

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"


## 2. Requisição HTTP
(Explorar manipulação de dicionário)

A API espera latitude/longitude, intervalo de datas, a lista de variáveis horárias
(`hourly`, separadas por vírgula) e um timezone — sem isso os horários vêm em UTC.

Escrevemos uma função pequena e testamos com **uma única cidade** antes de repetir
para todas — é mais fácil depurar um problema de request/parâmetro isolado do que
dentro de um loop.


In [3]:
def buscar_clima_historico(lat: float, lon: float, data_inicio: str, data_fim: str) -> dict:
    # Busca dados horários históricos de clima para um ponto (lat, lon)
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": data_inicio,
        "end_date": data_fim,
        "hourly": ",".join(VARIAVEIS_HORARIAS),
        "timezone": "America/Sao_Paulo",
    }
    resposta = requests.get(BASE_URL, params=params, timeout=30)
    resposta.raise_for_status()
    return resposta.json()


In [4]:
",".join(VARIAVEIS_HORARIAS)

'temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m'

In [ ]:
# Explorando o dicionario
print(CIDADES["sao_paulo"].items()) 
args = {k: v for k, v in CIDADES["sao_paulo"].items() if k in ("lat", "lon")}
print(args)
# print(lat=-23.5505, lon=-46.6333) na função da certo pois são as variáveis que ela espera receber

dict_items([('nome_exibicao', 'São Paulo'), ('lat', -23.5505), ('lon', -46.6333)])
{'lat': -23.5505, 'lon': -46.6333}


In [6]:
# Teste com uma única cidade antes de repetir para todas
# No ** antes de um dicionário faz desempacotamento de argumentos nomeados (**kwargs) — ele "abre" o dicionário e passa cada par chave: valor como um argumento chave=valor na chamada da função.
teste = buscar_clima_historico(**{k: v for k, v in CIDADES["sao_paulo"].items() if k in ("lat", "lon")},
                                data_inicio=DATA_INICIO, data_fim=DATA_FIM)
list(teste.keys())


['latitude',
 'longitude',
 'generationtime_ms',
 'utc_offset_seconds',
 'timezone',
 'timezone_abbreviation',
 'elevation',
 'hourly_units',
 'hourly']

In [17]:
teste

{'latitude': -23.514938,
 'longitude': -46.610504,
 'generationtime_ms': 15.299081802368164,
 'utc_offset_seconds': -10800,
 'timezone': 'America/Sao_Paulo',
 'timezone_abbreviation': 'GMT-3',
 'elevation': 758.0,
 'hourly_units': {'time': 'iso8601',
  'temperature_2m': '°C',
  'relative_humidity_2m': '%',
  'precipitation': 'mm',
  'wind_speed_10m': 'km/h'},
 'hourly': {'time': ['2025-01-01T00:00',
   '2025-01-01T01:00',
   '2025-01-01T02:00',
   '2025-01-01T03:00',
   '2025-01-01T04:00',
   '2025-01-01T05:00',
   '2025-01-01T06:00',
   '2025-01-01T07:00',
   '2025-01-01T08:00',
   '2025-01-01T09:00',
   '2025-01-01T10:00',
   '2025-01-01T11:00',
   '2025-01-01T12:00',
   '2025-01-01T13:00',
   '2025-01-01T14:00',
   '2025-01-01T15:00',
   '2025-01-01T16:00',
   '2025-01-01T17:00',
   '2025-01-01T18:00',
   '2025-01-01T19:00',
   '2025-01-01T20:00',
   '2025-01-01T21:00',
   '2025-01-01T22:00',
   '2025-01-01T23:00',
   '2025-01-02T00:00',
   '2025-01-02T01:00',
   '2025-01-02T02:00',

## 3. Explorando o JSON de resposta
(Explorar o conceito de json aninhado)

Repare na estrutura: metadados no nível raiz (`latitude`, `timezone`, `elevation`...) e,
dentro de `hourly`, **arrays paralelos** — `time[i]`, `temperature_2m[i]`,
`relative_humidity_2m[i]` etc. se referem todos à mesma i-ésima hora. Não existe uma
lista de "registros"; é uma lista de colunas.


In [ ]:
print("Chaves de nível raiz:", list(teste.keys()))
print("\nUnidades de cada variável horária:")
print(json.dumps(teste["hourly_units"], indent=2, ensure_ascii=False)) #é apenas um metadado. O hourly é o dado em si


Chaves de nível raiz: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']

Unidades de cada variável horária:
{
  "time": "iso8601",
  "temperature_2m": "°C",
  "relative_humidity_2m": "%",
  "precipitation": "mm",
  "wind_speed_10m": "km/h"
}


In [8]:
# Os arrays paralelos precisam ter o mesmo tamanho — é essa premissa que
# nos permitirá montar um DataFrame linha a linha na próxima parte.
for chave, valores in teste["hourly"].items():
    print(f"{chave}: {len(valores)} valores")


time: 744 valores
temperature_2m: 744 valores
relative_humidity_2m: 744 valores
precipitation: 744 valores
wind_speed_10m: 744 valores


In [19]:
teste['hourly'].keys()

dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m'])

In [9]:
# Espiando as primeiras horas
for campo in ["time", "temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]:
    print(f"{campo}: {teste['hourly'][campo][:5]}")


time: ['2025-01-01T00:00', '2025-01-01T01:00', '2025-01-01T02:00', '2025-01-01T03:00', '2025-01-01T04:00']
temperature_2m: [19.7, 19.3, 19.0, 18.6, 18.3]
relative_humidity_2m: [92, 94, 95, 97, 98]
precipitation: [0.0, 0.0, 0.0, 0.0, 0.0]
wind_speed_10m: [8.0, 7.5, 7.0, 6.9, 6.5]


## 4. Buscando as 5 cidades

Agora repetimos a chamada para todas as cidades. Adicionamos um pequeno `sleep` entre
requisições — não é estritamente necessário para 5 chamadas, mas é uma boa prática ao
consumir APIs públicas gratuitas (evita sobrecarregar o serviço).


In [10]:
dados_brutos = {}

for slug, info in CIDADES.items():
    print(f"Buscando {info['nome_exibicao']}...")
    dados_brutos[slug] = buscar_clima_historico(info["lat"], info["lon"], DATA_INICIO, DATA_FIM)
    time.sleep(0.5)

print("\nCidades obtidas:", list(dados_brutos.keys()))


Buscando São Paulo...
Buscando Rio de Janeiro...
Buscando Manaus...
Buscando Porto Alegre...
Buscando Recife...

Cidades obtidas: ['sao_paulo', 'rio_de_janeiro', 'manaus', 'porto_alegre', 'recife']


Abordar data quality e se não tiver o mesmo tamanho vai ter problema em transformar em data frame

In [11]:
# Checagem de consistência: mesmo número de horas em todas as cidades?
for slug, payload in dados_brutos.items():
    n_horas = len(payload["hourly"]["time"])
    print(f"{slug:>15s}: {n_horas} horas — primeira={payload['hourly']['time'][0]} última={payload['hourly']['time'][-1]}")


      sao_paulo: 744 horas — primeira=2025-01-01T00:00 última=2025-01-31T23:00
 rio_de_janeiro: 744 horas — primeira=2025-01-01T00:00 última=2025-01-31T23:00
         manaus: 744 horas — primeira=2025-01-01T00:00 última=2025-01-31T23:00
   porto_alegre: 744 horas — primeira=2025-01-01T00:00 última=2025-01-31T23:00
         recife: 744 horas — primeira=2025-01-01T00:00 última=2025-01-31T23:00


## 5. Salvando o JSON bruto em `data/raw`

Salvamos **um arquivo por cidade** (fiel ao que a API devolveu, sem nenhum tratamento)
mais um `manifest.json` com os metadados da coleta. Isso separa claramente "dado como
a fonte nos entregou" de qualquer transformação — a próxima parte começa exatamente
daqui.


In [12]:
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

for slug, payload in dados_brutos.items():
    caminho = RAW_DIR / f"clima_raw_{slug}.json"
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    print(f"Salvo: {caminho} ({caminho.stat().st_size / 1024:.1f} KB)")


Salvo: ../data/raw/clima_raw_sao_paulo.json (29.7 KB)
Salvo: ../data/raw/clima_raw_rio_de_janeiro.json (29.7 KB)
Salvo: ../data/raw/clima_raw_manaus.json (29.6 KB)
Salvo: ../data/raw/clima_raw_porto_alegre.json (29.8 KB)
Salvo: ../data/raw/clima_raw_recife.json (29.9 KB)


In [13]:
manifesto = {
    "data_inicio": DATA_INICIO,
    "data_fim": DATA_FIM,
    "variaveis_horarias": VARIAVEIS_HORARIAS,
    "cidades": CIDADES,
}

with open(RAW_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifesto, f, ensure_ascii=False, indent=2)

print("Manifesto salvo em", RAW_DIR / "manifest.json")


Manifesto salvo em ../data/raw/manifest.json


## 6. Conferência final

Antes de fechar o notebook, confirmamos que todos os arquivos esperados existem no
disco — é um hábito simples que evita começar a próxima parte em cima de um arquivo
corrompido ou faltando.


In [14]:
for arquivo in sorted(RAW_DIR.iterdir()):
    print(arquivo.name)


clima_raw_manaus.json
clima_raw_porto_alegre.json
clima_raw_recife.json
clima_raw_rio_de_janeiro.json
clima_raw_sao_paulo.json
manifest.json
